# 01.7 Training Loop

This notebook is where the previous notebooks finally come together.

You have already learned about:

- tensors
- autograd
- datasets and data loaders
- model definition
- loss functions and optimizers

A training loop organizes all of those pieces in the correct order.


## Learning Goals

After this notebook, you should be able to:

1. Explain the steps of a full training loop.
2. Use `model.train()` and `model.eval()` correctly.
3. Organize `zero_grad()`, `backward()`, and `step()` correctly.
4. Distinguish training from validation.
5. Track loss and accuracy.
6. Write a reusable minimal training framework.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## Prepare a Minimal Classification Dataset

To focus on the training loop itself, we directly construct a 2D binary classification dataset.


In [ ]:
torch.manual_seed(0)

class0 = torch.randn(80, 2) * 0.6 + torch.tensor([-1.2, -1.0])
class1 = torch.randn(80, 2) * 0.6 + torch.tensor([1.2, 1.0])

X = torch.cat([class0, class1], dim=0).float()
y = torch.cat([
    torch.zeros(len(class0), dtype=torch.long),
    torch.ones(len(class1), dtype=torch.long),
])

perm = torch.randperm(len(X))
X = X[perm]
y = y[perm]

split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)
print("X_val.shape =", X_val.shape)
print("y_val.shape =", y_val.shape)

In [ ]:
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)
print("first 3 labels:", yb[:3])

## Define the Model

Here we use a very simple MLP.


In [ ]:
class SmallClassifier(nn.Module):
    def __init__(self, in_features=2, hidden_features=8, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(),
            nn.Linear(hidden_features, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = SmallClassifier()
print(model)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print(loss_fn)
print(optimizer)

## The Minimal Skeleton of a Training Loop

The standard order in the training phase is usually:

1. `model.train()`
2. iterate over batches
3. `optimizer.zero_grad()`
4. `pred = model(xb)`
5. `loss = loss_fn(pred, yb)`
6. `loss.backward()`
7. `optimizer.step()`

The validation phase is usually:

1. `model.eval()`
2. `with torch.no_grad():`
3. forward only, no parameter updates

In [ ]:
def compute_accuracy(logits, targets):
    preds = logits.argmax(dim=1)
    acc = (preds == targets).float().mean().item()
    return acc


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += compute_accuracy(logits, yb)
        num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    num_batches = 0

    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            total_loss += loss.item()
            total_acc += compute_accuracy(logits, yb)
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches

## Train for a Few Epochs

An `epoch` means one full pass over the training set.


In [ ]:
history = []

for epoch in range(1, 16):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        }
    )

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"epoch={epoch:02d} | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

In [ ]:
print("last history record / last history record:")
print(history[-1])

## Why `train()` and `eval()` Matter

This small model does not have `Dropout` or `BatchNorm`, so the difference is not dramatic yet.

But as an engineering habit, these two modes must be used correctly.

- training mode
- evaluation mode

## Mini Exercises

The point of these exercises is not writing lots of code, but truly remembering the structure of the training loop.


In [ ]:
# Exercise 1
# In one sentence, explain why training usually starts with optimizer.zero_grad().


Reference answer:

Because `PyTorch` accumulates gradients by default, we usually clear the old gradients before each batch.


In [ ]:
# Exercise 2
# 
# Goal:
# Given logits and targets, return the batch accuracy.

def batch_accuracy(logits, targets):
    # TODO
    pass


# logits = torch.tensor([[2.0, 1.0], [0.2, 0.8], [1.5, 0.1]])
# targets = torch.tensor([0, 1, 0])
# print(batch_accuracy(logits, targets))

In [ ]:
# Exercise 2 Reference Solution

def batch_accuracy_solution(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


logits = torch.tensor([[2.0, 1.0], [0.2, 0.8], [1.5, 0.1]])
targets = torch.tensor([0, 1, 0])
print(batch_accuracy_solution(logits, targets))

In [ ]:
# Exercise 3
# Explain what each of the two lines below does.
#
# model.train()
# model.eval()
#Write two sentences in your own words.
# Write 2 sentences in your own words.

## Summary

The core order of the training loop should become muscle memory:

1. get a batch
2. forward pass
3. compute loss
4. zero gradients
5. backward pass
6. update parameters

You should now be able to answer:

1. Why is training code different from validation code?
2. Why should the order of `zero_grad()`, `backward()`, and `step()` not be changed casually?
3. What does one epoch mean?

Suggested next step:

- Move to the saving-and-inference notebook to save and reuse trained models.